#  Griffin Library - Intelligent Book Recommendation System

## Notebook 02 · Data Cleaning & Merging

**Goal:** Clean the primary dataset (Best Books 10k), enrich it with CMU summaries  
where available, and produce a single high-quality merged dataset ready for EDA.

| | Details |
|---|---|
| **Input** | `data/raw/goodreads_data.csv` · `data/raw/booksummaries.txt` |
| **Operations** | Filtering · Cleaning · Genre parsing · CMU enrichment (exact + fuzzy) · Tier assignment |
| **Output** | `data/processed/books_merged.csv` |
| **Next Step** | `03_eda.ipynb` - Explore the clean merged dataset |

---

In [1]:
import sys
# Dependencies installed via requirements.txt

import pandas as pd
import os

os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

# ── Load Goodreads ──────────────────────────────────────────
gr = pd.read_csv("data/raw/books.csv", on_bad_lines="skip")

# ── Load CMU ────────────────────────────────────────────────
cmu = pd.read_csv(
    "data/raw/booksummaries.txt",
    sep="\t",
    header=None,
    names=["wiki_id","freebase_id","title","author","pub_date","genres","summary"],
    on_bad_lines="skip",
)

print(f"Goodreads : {gr.shape}")
print(f"CMU       : {cmu.shape}")

Goodreads : (11123, 12)
CMU       : (16559, 7)


In [2]:
# ═══════════════════════════════════════════════════════════════
# CLEAN GOODREADS
# ═══════════════════════════════════════════════════════════════

# Keep English only
gr_clean = gr[gr["language_code"].isin(["eng", "en-US", "en-GB"])].copy()

# Drop books with no real ratings
gr_clean = gr_clean[gr_clean["average_rating"] > 0]
gr_clean = gr_clean[gr_clean["ratings_count"] >= 10]

# Fix column name (has leading space)
gr_clean = gr_clean.rename(columns={"  num_pages": "num_pages"})

# Keep relevant columns only
gr_clean = gr_clean[[
    "bookID", "title", "authors", "average_rating",
    "ratings_count", "text_reviews_count",
    "num_pages", "publication_date", "isbn"
]].drop_duplicates(subset="title")

gr_clean = gr_clean.reset_index(drop=True)

print(f"Before cleaning : {len(gr):>6,}")
print(f"After cleaning  : {len(gr_clean):>6,}")
print(f"Removed         : {len(gr) - len(gr_clean):>6,}")

Before cleaning : 11,123
After cleaning  :  9,325
Removed         :  1,798


### Clean Goodreads

Three filters applied to remove low-quality entries:
- **Non-English books** → dropped (~900 books in Spanish, French, German, etc.)
- **Books with `average_rating = 0`** → no real user ratings, unusable as ranking signal
- **Books with `ratings_count < 10`** → too few ratings to trust the score
- **Duplicate titles** → keep first occurrence only

Result: **9,325 reliable, English, well-rated books** ready for merging.

---

In [3]:
# ═══════════════════════════════════════════════════════════════
# CLEAN CMU
# ═══════════════════════════════════════════════════════════════

import json

def parse_genres(genre_str):
    """Parse CMU genres from JSON string to clean comma-separated list."""
    if pd.isna(genre_str):
        return ""
    try:
        genre_dict = json.loads(genre_str)
        return ", ".join(genre_dict.values())
    except:
        return ""

# Parse genres
cmu_clean = cmu.copy()
cmu_clean["genres_parsed"] = cmu_clean["genres"].apply(parse_genres)

# Keep relevant columns only
cmu_clean = cmu_clean[[
    "title", "author", "summary", "genres_parsed"
]].drop_duplicates(subset="title")

cmu_clean = cmu_clean.reset_index(drop=True)

print(f"Before cleaning : {len(cmu):>6,}")
print(f"After cleaning  : {len(cmu_clean):>6,}")
print(f"\nGenre sample:")
print(cmu_clean["genres_parsed"].iloc[0])
print(f"\nSummary sample (300 chars):")
print(cmu_clean["summary"].iloc[0][:300])

Before cleaning : 16,559
After cleaning  : 16,277

Genre sample:
Roman à clef, Satire, Children's literature, Speculative fiction, Fiction

Summary sample (300 chars):
 Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the animals a revolutionary song, 'Beasts of England'. When Major dies, two young pigs, Snowball and Napoleon, assume command and turn his dream into a philo


### Clean CMU

- **Genres parsed**: converted from raw JSON `{"/m/016lj8": "Roman à clef"}` to clean string `"Roman à clef, Satire, Fiction"`
- **Duplicate titles removed**: 282 duplicates dropped
- **Kept**: `title`, `author`, `summary`, `genres_parsed` - everything else discarded

Result: **16,277 books with clean genres and rich summaries** ready for merging.

---

In [4]:
# ═══════════════════════════════════════════════════════════════
# MERGE - Exact Title Match
# ═══════════════════════════════════════════════════════════════

# Normalize titles for matching
gr_clean["title_lower"] = gr_clean["title"].str.strip().str.lower()
cmu_clean["title_lower"] = cmu_clean["title"].str.strip().str.lower()

# Exact merge
merged_exact = gr_clean.merge(
    cmu_clean[["title_lower", "summary", "genres_parsed"]],
    on="title_lower",
    how="left"
)

exact_matched = merged_exact["summary"].notna().sum()
exact_missing = merged_exact["summary"].isna().sum()

print(f"Total books      : {len(merged_exact):>6,}")
print(f"Exact matches    : {exact_matched:>6,}")
print(f"No match yet     : {exact_missing:>6,}")
print(f"Match rate       : {exact_matched/len(merged_exact)*100:.1f}%")

Total books      :  9,326
Exact matches    :  1,266
No match yet     :  8,060
Match rate       : 13.6%


### Merge - Exact Title Match

- **1,266 books** matched with full CMU summary via exact title match
- **8,060 books** still without summary → will attempt fuzzy matching next
---

In [5]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], capture_output=True)
print("rapidfuzz ready")

rapidfuzz ready


In [6]:
import sys
# Dependencies installed via requirements.txt

import subprocess
subprocess.run([
    "-m", "pip", "install", "rapidfuzz"
], capture_output=False)

CompletedProcess(args=['C:\\Users\\suras\\Desktop\\EverQuote-V2\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'rapidfuzz'], returncode=0)

In [7]:
import sys
# Dependencies installed via requirements.txt

from rapidfuzz import process, fuzz

cmu_titles_list = cmu_clean["title_lower"].tolist()

# Only run fuzzy on unmatched books
unmatched_mask = merged_exact["summary"].isna()
unmatched_titles = merged_exact.loc[unmatched_mask, "title_lower"].tolist()

print(f"Running fuzzy match on {len(unmatched_titles):,} titles...")

fuzzy_summaries = []
fuzzy_genres    = []

for title in unmatched_titles:
    result = process.extractOne(
        title,
        cmu_titles_list,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=88
    )
    if result:
        matched_row = cmu_clean[cmu_clean["title_lower"] == result[0]].iloc[0]
        fuzzy_summaries.append(matched_row["summary"])
        fuzzy_genres.append(matched_row["genres_parsed"])
    else:
        fuzzy_summaries.append(None)
        fuzzy_genres.append(None)

# Fill in fuzzy matches
merged_exact.loc[unmatched_mask, "summary"]       = fuzzy_summaries
merged_exact.loc[unmatched_mask, "genres_parsed"] = fuzzy_genres

fuzzy_matched = merged_exact["summary"].notna().sum()
still_missing = merged_exact["summary"].isna().sum()

print(f"\nAfter fuzzy matching:")
print(f"Total matched    : {fuzzy_matched:>6,}")
print(f"Still unmatched  : {still_missing:>6,}")
print(f"Match rate       : {fuzzy_matched/len(merged_exact)*100:.1f}%")

Running fuzzy match on 8,060 titles...

After fuzzy matching:
Total matched    :  1,411
Still unmatched  :  7,915
Match rate       : 15.1%


### Merge - Exact + Fuzzy Title Match (vs Original Goodreads)

- **Exact match**: 1,266 books matched with CMU summary
- **Fuzzy match**: added only 145 books (score cutoff ≥ 88)
- **Final match rate**: 15.1% - critically low coverage
- **Root cause**: original Goodreads dataset (11k books) has no description column, and CMU overlap is structurally limited
- **Decision**: discard original `books.csv` as primary dataset
- **New strategy**: adopt `Best Books 10k` as primary dataset - it contains Goodreads descriptions natively, eliminating the merge dependency on CMU entirely
- **CMU role going forward**: enrichment only - longer narrative summaries merged on top where title match exists

---

In [8]:
import sys
# Dependencies installed via requirements.txt

import pandas as pd
import os
import json
import re

os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

# ── Load Primary Dataset (Best Books 10k) ───────────────────
df = pd.read_csv("data/raw/goodreads_data.csv")

# ── Load CMU (Enrichment Layer) ──────────────────────────────
cmu = pd.read_csv(
    "data/raw/booksummaries.txt",
    sep="\t",
    header=None,
    names=["wiki_id","freebase_id","title","author","pub_date","genres","summary"],
    on_bad_lines="skip",
)

print(f"Primary dataset : {df.shape}")
print(f"CMU enrichment  : {cmu.shape}")
print(f"\nPrimary columns : {list(df.columns)}")

Primary dataset : (10000, 8)
CMU enrichment  : (16559, 7)

Primary columns : ['Unnamed: 0', 'Book', 'Author', 'Description', 'Genres', 'Avg_Rating', 'Num_Ratings', 'URL']


---

In [9]:
# ═══════════════════════════════════════════════════════════════
# Clean Primary Dataset (Best Books 10k)
# ═══════════════════════════════════════════════════════════════

# Rename columns to standard names
df = df.rename(columns={
    "Book"        : "title",
    "Author"      : "authors",
    "Description" : "description",
    "Genres"      : "genres_raw",
    "Avg_Rating"  : "avg_rating",
    "Num_Ratings" : "num_ratings",
    "URL"         : "url",
}).drop(columns=["Unnamed: 0"])

# Convert num_ratings from string "5,691,311" to integer
df["num_ratings"] = (
    df["num_ratings"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
)

# Drop nulls in critical columns
df = df.dropna(subset=["title", "description", "avg_rating"])

# Drop low-quality entries
df = df[df["avg_rating"] > 0]
df = df[df["num_ratings"] >= 50]

# Deduplicate
df = df.drop_duplicates(subset="title").reset_index(drop=True)

# Add unique book_id
df["book_id"] = df.index + 1

print(f"After cleaning  : {len(df):>6,}")
print(f"\nSample:")
print(df[["title","authors","avg_rating","num_ratings"]].head(3))

After cleaning  :  8,577

Sample:
                                               title       authors  \
0                              To Kill a Mockingbird    Harper Lee   
1  Harry Potter and the Philosopher’s Stone (Harr...  J.K. Rowling   
2                                Pride and Prejudice   Jane Austen   

   avg_rating  num_ratings  
0        4.27      5691311  
1        4.47      9278135  
2        4.28      3944155  


### Clean Primary Dataset

- Renamed all columns to consistent snake_case names
- Converted `num_ratings` from string format `"5,691,311"` to integer
- Dropped 77 books with null descriptions
- Dropped books with `avg_rating = 0` or `num_ratings < 50`
- Removed duplicate titles

Result: **8,577 clean, well-rated books** ready for genre parsing and enrichment.

---

In [10]:
# ═══════════════════════════════════════════════════════════════
# Parse Genres
# ═══════════════════════════════════════════════════════════════

import ast

def parse_genres(genre_str):
    """Convert string list "['Classics', 'Fiction']" to clean string."""
    if pd.isna(genre_str):
        return ""
    try:
        genres = ast.literal_eval(genre_str)
        return ", ".join(g.strip() for g in genres if g.strip())
    except:
        return str(genre_str).strip()

df["genres"] = df["genres_raw"].apply(parse_genres)
df = df.drop(columns=["genres_raw"])

print("Genre parsing complete.")
print(f"\nSample genres:")
for g in df["genres"].head(5):
    print(f"  • {g}")

print(f"\nBooks with no genres: {(df['genres'] == '').sum()}")

Genre parsing complete.

Sample genres:
  • Classics, Fiction, Historical Fiction, School, Literature, Young Adult, Historical
  • Fantasy, Fiction, Young Adult, Magic, Childrens, Middle Grade, Classics
  • Classics, Fiction, Romance, Historical Fiction, Literature, Historical, Audiobook
  • Classics, Nonfiction, History, Biography, Memoir, Historical, Holocaust
  • Classics, Fiction, Dystopia, Fantasy, Politics, School, Literature

Books with no genres: 134


### Parse Genres

- Converted genres from Python string list format to clean comma-separated strings
- 134 books have no genres - will use description only for embedding (acceptable)

Result: **Rich multi-genre tags** ready for filtering and embedding text construction.

---

In [11]:
# ═══════════════════════════════════════════════════════════════
# Clean CMU (Enrichment Layer)
# ═══════════════════════════════════════════════════════════════

def parse_cmu_genres(genre_str):
    if pd.isna(genre_str):
        return ""
    try:
        genre_dict = json.loads(genre_str)
        return ", ".join(genre_dict.values())
    except:
        return ""

cmu_clean = cmu[["title", "author", "summary", "genres"]].copy()
cmu_clean["genres"] = cmu_clean["genres"].apply(parse_cmu_genres)
cmu_clean = cmu_clean.dropna(subset=["summary"])
cmu_clean = cmu_clean.drop_duplicates(subset="title").reset_index(drop=True)

# Normalize titles for matching
cmu_clean["title_lower"] = cmu_clean["title"].str.strip().str.lower()

print(f"CMU ready       : {len(cmu_clean):>6,} books")
print(f"\nSample summary (300 chars):")
print(cmu_clean["summary"].iloc[0][:300])

CMU ready       : 16,277 books

Sample summary (300 chars):
 Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the animals a revolutionary song, 'Beasts of England'. When Major dies, two young pigs, Snowball and Napoleon, assume command and turn his dream into a philo


---

In [12]:
# ═══════════════════════════════════════════════════════════════
# Merge CMU Summaries (Exact + Fuzzy)
# ═══════════════════════════════════════════════════════════════

from rapidfuzz import process, fuzz

# Normalize titles
df["title_lower"] = df["title"].str.strip().str.lower()
cmu_titles_list   = cmu_clean["title_lower"].tolist()

# ── Exact Match ──────────────────────────────────────────────
df = df.merge(
    cmu_clean[["title_lower", "summary"]],
    on="title_lower",
    how="left"
)

exact_matched = df["summary"].notna().sum()
print(f"Exact matches   : {exact_matched:>6,} / {len(df):,}")

# ── Fuzzy Match (unmatched only) ─────────────────────────────
unmatched_mask   = df["summary"].isna()
unmatched_titles = df.loc[unmatched_mask, "title_lower"].tolist()

print(f"Running fuzzy on {len(unmatched_titles):,} unmatched titles...")

fuzzy_summaries = []
for title in unmatched_titles:
    result = process.extractOne(
        title,
        cmu_titles_list,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=88
    )
    if result:
        matched = cmu_clean[cmu_clean["title_lower"] == result[0]].iloc[0]
        fuzzy_summaries.append(matched["summary"])
    else:
        fuzzy_summaries.append(None)

df.loc[unmatched_mask, "summary"] = fuzzy_summaries

total_matched = df["summary"].notna().sum()
still_missing = df["summary"].isna().sum()

print(f"\nExact matched   : {exact_matched:>6,}")
print(f"Fuzzy matched   : {total_matched - exact_matched:>6,}")
print(f"Total matched   : {total_matched:>6,} ({total_matched/len(df)*100:.1f}%)")
print(f"No CMU summary  : {still_missing:>6,} ({still_missing/len(df)*100:.1f}%)")

Exact matches   :  1,476 / 8,577
Running fuzzy on 7,101 unmatched titles...

Exact matched   :  1,476
Fuzzy matched   :    108
Total matched   :  1,584 (18.5%)
No CMU summary  :  6,993 (81.5%)


### CMU Enrichment Merge (Exact + Fuzzy)

- **Exact match**: 1,476 books enriched with full CMU narrative summary
- **Fuzzy match**: 108 additional books matched (score cutoff ≥ 88)
- **Total enriched**: 1,584 books (18.5%) - have both Goodreads description + CMU summary
- **Remaining 81.5%**: use Goodreads description only - still high quality for embedding
- **Key insight**: CMU enrichment is a bonus, not a dependency - Best Books 10k descriptions are sufficient as standalone embedding text

---

In [13]:
# ═══════════════════════════════════════════════════════════════
# Build Embedding Text Column
# ═══════════════════════════════════════════════════════════════

def build_embedding_text(row):
    """
    Combine all available signals into a single rich text for embedding.
    Priority: CMU summary (if available) + Goodreads description + genres
    """
    parts = []

    # Title and author always included
    parts.append(f"{row['title']} by {row['authors']}")

    # Genres
    if row["genres"]:
        parts.append(row["genres"])

    # Goodreads description (always available)
    if pd.notna(row["description"]) and str(row["description"]).strip():
        parts.append(str(row["description"]).strip())

    # CMU summary (enrichment — longer narrative)
    if pd.notna(row["summary"]) and str(row["summary"]).strip():
        parts.append(str(row["summary"]).strip()[:1000])

    return " | ".join(parts)

df["embedding_text"] = df.apply(build_embedding_text, axis=1)

# Stats
has_cmu    = df["summary"].notna().sum()
no_cmu     = df["summary"].isna().sum()
avg_length = df["embedding_text"].str.len().mean()

print(f"Embedding text built for all {len(df):,} books")
print(f"\nWith CMU summary    : {has_cmu:,} books (richer embedding)")
print(f"Without CMU summary : {no_cmu:,} books (description only)")
print(f"Avg text length     : {avg_length:.0f} chars")
print(f"\nSample (Tier 1 - with CMU):")
print(df[df['summary'].notna()]['embedding_text'].iloc[0][:400])
print(f"\nSample (Tier 2 - description only):")
print(df[df['summary'].isna()]['embedding_text'].iloc[0][:400])

Embedding text built for all 8,577 books

With CMU summary    : 1,584 books (richer embedding)
Without CMU summary : 6,993 books (description only)
Avg text length     : 1251 chars

Sample (Tier 1 - with CMU):
To Kill a Mockingbird by Harper Lee | Classics, Fiction, Historical Fiction, School, Literature, Young Adult, Historical | The unforgettable novel of a childhood in a sleepy Southern town and the crisis of conscience that rocked it. "To Kill A Mockingbird" became both an instant bestseller and a critical success when it was first published in 1960. It went on to win the Pulitzer Prize in 1961 and 

Sample (Tier 2 - description only):
Harry Potter and the Philosopher’s Stone (Harry Potter, #1) by J.K. Rowling | Fantasy, Fiction, Young Adult, Magic, Childrens, Middle Grade, Classics | Harry Potter thinks he is an ordinary boy - until he is rescued by an owl, taken to Hogwarts School of Witchcraft and Wizardry, learns to play Quidditch and does battle in a deadly duel. The Reason ..

### Build Embedding Text

- Combined all available signals: `title + authors + genres + description + CMU summary`
- Average embedding text length: **1,251 characters** — rich enough for high-quality semantic search
- **Tier 1** (18.5%): title + genres + Goodreads description + CMU narrative summary
- **Tier 2** (81.5%): title + genres + Goodreads description only - still highly informative

Result: **Every book has a meaningful embedding text** - no book relies on genres alone.

---

In [14]:
# ═══════════════════════════════════════════════════════════════
# Assign Tiers & Save
# ═══════════════════════════════════════════════════════════════

# Assign tier
df["tier"] = df["summary"].apply(lambda x: 1 if pd.notna(x) else 2)

# ── Bring num_pages from gr_clean (books.csv) via title match ──
gr_pages = gr_clean[["title", "num_pages"]].copy()
gr_pages["title_lower"] = gr_pages["title"].str.strip().str.lower()
df["title_lower_tmp"]   = df["title"].str.strip().str.lower()

pages_map  = gr_pages.drop_duplicates("title_lower").set_index("title_lower")["num_pages"]
df["num_pages"] = (
    df["title_lower_tmp"]
    .map(pages_map)
    .fillna(0)
    .astype(int)
)
df.drop(columns=["title_lower_tmp"], inplace=True)

print(f"Books with page count : {(df['num_pages'] > 0).sum():,} / {len(df):,}")

# Final column selection
df_final = df[[
    "book_id", "title", "authors", "genres",
    "avg_rating", "num_ratings", "description",
    "summary", "embedding_text", "tier", "url",
    "num_pages",
]].copy()

# Save as CSV
os.makedirs("data/processed", exist_ok=True)
df_final.to_csv("data/processed/books_merged.csv", index=False)

print(f"Saved: data/processed/books_merged.csv")
print(f"\nFinal dataset summary:")
print(f"  Total books     : {len(df_final):>6,}")
print(f"  Tier 1 (+ CMU)  : {(df_final['tier']==1).sum():>6,}")
print(f"  Tier 2 (desc)   : {(df_final['tier']==2).sum():>6,}")
print(f"  Avg rating      : {df_final['avg_rating'].mean():.2f}")
print(f"  Columns         : {list(df_final.columns)}")


Books with page count : 1,084 / 8,577
Saved: data/processed/books_merged.csv

Final dataset summary:
  Total books     :  8,577
  Tier 1 (+ CMU)  :  1,584
  Tier 2 (desc)   :  6,993
  Avg rating      : 4.04
  Columns         : ['book_id', 'title', 'authors', 'genres', 'avg_rating', 'num_ratings', 'description', 'summary', 'embedding_text', 'tier', 'url', 'num_pages']


###  Assign Tiers & Save

- **Tier 1** (1,584 books): Goodreads description + CMU narrative summary - richest embedding
- **Tier 2** (6,993 books): Goodreads description only - sufficient for high-quality embedding
- **Average rating**: 4.04 - high quality catalog overall
- **Output**: `data/processed/books_merged.csv` - 8,577 books, 11 columns

Result: **Single clean dataset ready for EDA and feature engineering.**